# Logistic regression learning curves

This notebook measures how the TF-IDF logistic regression classifier's training and validation accuracy change as the training set grows. The validation set stays fixed while a fresh pipeline is fitted on progressively larger training subsets.

The hyperparameters are the best settings found in the baseline notebook. Calibration is omitted because it does not affect the learning-curve class predictions and fitting calibration on the validation set would make that split part of training. The test split is not used.


In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from matplotlib.ticker import PercentFormatter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline


## Load the training and validation data


In [ ]:
dataset = load_dataset("rasbt/human-vs-ai-50k")
train_dataset = dataset["train"]
validation_dataset = dataset["validation"]

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "human": split["label"].count(0),
            "ai": split["label"].count(1),
            "total": len(split),
        }
        for name, split in {
            "train": train_dataset,
            "validation": validation_dataset,
        }.items()
    ]
).set_index("split")

split_summary


## Construct nested training subsets

The subsets contain 1%, 2.5%, 5%, 10%, 25%, 50%, and 100% of the training data. A distribution-aware ordering keeps the joint label and source-collection distribution approximately stable. Each larger subset contains all samples from the preceding smaller subset.


In [ ]:
RANDOM_STATE = 17
TRAIN_FRACTIONS = np.asarray(
    [0.01, 0.025, 0.05, 0.10, 0.25, 0.50, 1.00]
)


In [ ]:
def make_distribution_aware_order(
    labels, source_collections, seed
):
    labels = np.asarray(labels, dtype=np.int64)
    source_collections = np.asarray(source_collections, dtype=str)
    strata = np.asarray(
        [
            f"{label}::{source}"
            for label, source in zip(labels, source_collections)
        ]
    )
    rng = np.random.default_rng(seed)
    groups = {}
    for stratum in sorted(np.unique(strata)):
        indices = np.flatnonzero(strata == stratum)
        rng.shuffle(indices)
        groups[stratum] = indices.tolist()

    total = labels.size
    proportions = {
        stratum: len(indices) / total
        for stratum, indices in groups.items()
    }
    selected = {stratum: 0 for stratum in groups}
    order = []
    for step in range(total):
        available = [
            stratum
            for stratum, indices in groups.items()
            if selected[stratum] < len(indices)
        ]
        stratum = max(
            available,
            key=lambda name: (
                proportions[name] * (step + 1) - selected[name],
                name,
            ),
        )
        order.append(groups[stratum][selected[stratum]])
        selected[stratum] += 1

    assert sorted(order) == list(range(total))
    return np.asarray(order, dtype=np.int64)


nested_order = make_distribution_aware_order(
    labels=train_dataset["label"],
    source_collections=train_dataset["source_collection"],
    seed=RANDOM_STATE,
)
train_sizes = np.rint(
    TRAIN_FRACTIONS * len(train_dataset)
).astype(np.int64)
train_sizes = np.unique(
    np.clip(train_sizes, 2, len(train_dataset))
)
train_sizes[-1] = len(train_dataset)

previous_indices = set()
for sample_count in train_sizes:
    current_indices = set(nested_order[:sample_count].tolist())
    assert previous_indices.issubset(current_indices)
    previous_indices = current_indices

subset_summary = pd.DataFrame(
    [
        {
            "training_samples": int(sample_count),
            "training_data_used": sample_count / len(train_dataset),
            "ai_share": np.mean(
                np.asarray(train_dataset["label"])[
                    nested_order[:sample_count]
                ]
            ),
        }
        for sample_count in train_sizes
    ]
)
subset_summary.style.format(
    {
        "training_data_used": "{:.1%}",
        "ai_share": "{:.1%}",
    }
)


## Fit one fresh pipeline per training-set size

Each point starts with a new TF-IDF vectorizer and logistic regression classifier. The vocabulary and feature weights are therefore learned only from the selected training subset.


In [ ]:
def build_model():
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    max_df=0.98,
                    max_features=200_000,
                    min_df=2,
                    ngram_range=(1, 2),
                    sublinear_tf=True,
                    dtype=np.float32,
                ),
            ),
            (
                "logreg",
                LogisticRegression(
                    C=3.0,
                    max_iter=1_000,
                    random_state=RANDOM_STATE,
                    solver="liblinear",
                ),
            ),
        ]
    )


def find_project_dir():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scripts" / "13_learning-curves").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the ai-detector-from-scratch project")


PROJECT_DIR = find_project_dir()
STAGE_DIR = PROJECT_DIR / "scripts" / "13_learning-curves"
RESULTS_DIR = STAGE_DIR / "results"
FIGURES_DIR = STAGE_DIR / "figures"
RESULTS_PATH = RESULTS_DIR / "logreg-learning-curve-results.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results file: {RESULTS_PATH}")


In [ ]:
train_texts = np.asarray(train_dataset["text"], dtype=object)
train_labels = np.asarray(train_dataset["label"], dtype=np.int64)
validation_texts = list(validation_dataset["text"])
validation_labels = np.asarray(
    validation_dataset["label"], dtype=np.int64
)
results = []

for sample_count in train_sizes:
    sample_count = int(sample_count)
    train_indices = nested_order[:sample_count]
    subset_texts = train_texts[train_indices].tolist()
    subset_labels = train_labels[train_indices]

    model = build_model()
    print(f"\nTraining with {sample_count:,} samples")
    start_time = time.perf_counter()
    model.fit(subset_texts, subset_labels)
    training_runtime = time.perf_counter() - start_time

    training_accuracy = accuracy_score(
        subset_labels, model.predict(subset_texts)
    )
    validation_accuracy = accuracy_score(
        validation_labels, model.predict(validation_texts)
    )
    results.append(
        {
            "training_samples": sample_count,
            "training_fraction": sample_count / len(train_dataset),
            "training_accuracy": training_accuracy,
            "validation_accuracy": validation_accuracy,
            "generalization_gap": (
                training_accuracy - validation_accuracy
            ),
            "training_runtime_seconds": training_runtime,
        }
    )
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)
    print(
        f"Training accuracy: {training_accuracy:.2%}; "
        f"validation accuracy: {validation_accuracy:.2%}"
    )

curve_results = pd.DataFrame(results)
curve_results


## Plot the learning curves


In [ ]:
x_percent = curve_results["training_fraction"].to_numpy() * 100
training_accuracy = curve_results["training_accuracy"].to_numpy()
validation_accuracy = curve_results["validation_accuracy"].to_numpy()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(
    x_percent, training_accuracy,
    marker="o", linewidth=2, color="#1f77b4",
    label="Training accuracy",
)
ax.plot(
    x_percent, validation_accuracy,
    marker="o", linewidth=2, color="#666666",
    label="Validation accuracy",
)
ax.set_xscale("log")
ax.set_xticks(
    x_percent,
    [
        f"{fraction:.1f}%\n{samples:,}"
        for fraction, samples in zip(
            x_percent, curve_results["training_samples"]
        )
    ],
)
lower_limit = max(
    0.5,
    min(training_accuracy.min(), validation_accuracy.min()) - 0.03,
)
ax.set_ylim(lower_limit, 1.005)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Training data used")
ax.set_ylabel("Accuracy")
ax.set_title("Logistic regression learning curves", loc="left")
ax.grid(axis="y", color="#dddddd", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()

FIGURE_PATH = FIGURES_DIR / "logreg-learning-curves.svg"
fig.savefig(FIGURE_PATH, bbox_inches="tight")
plt.show()
print(f"Saved figure to {FIGURE_PATH}")


In [ ]:
curve_results.style.format(
    {
        "training_fraction": "{:.1%}",
        "training_accuracy": "{:.2%}",
        "validation_accuracy": "{:.2%}",
        "generalization_gap": "{:.2%}",
        "training_runtime_seconds": "{:,.1f}",
    }
)


## Interpreting the result

A large gap between training and validation accuracy suggests overfitting. If both curves remain low and close together, the model is more likely underfitting. If validation accuracy is still rising at the largest training-set sizes, collecting more training data may help. A flat validation curve suggests that additional samples from the same distribution may provide only small improvements.

These conclusions apply to the current data distribution and training configuration. A stronger robustness study could repeat every point with several random seeds and report the mean and standard deviation.
